# **Part 3: NLP and Sequence Modeling Mini Project**

In [1]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## Task 1: Dataset Understanding


#### Number of records


In [3]:
df = pd.read_csv("customer_support_text_classification.csv")

print(df.shape)

(1500, 6)


#### Target labels/classes


In [4]:
print(df['sentiment_label'].unique())

['neutral' 'positive' 'negative']


#### Sample text records


In [5]:
print(df['customer_message'].head())

0    I need information about the payment process. ...
1        I need information about the payment process.
2    The refund process was fast and convenient. I ...
3    My refund is still pending and this experience...
4     Please tell me how to update my account details.
Name: customer_message, dtype: object


#### Average text length


In [6]:
df['text_length'] = df['customer_message'].apply(len)

print(df['text_length'].mean())

72.75666666666666


#### Class distribution


In [7]:
print(df['sentiment_label'].value_counts())

sentiment_label
neutral     524
negative    497
positive    479
Name: count, dtype: int64


## Task 2: Text Preprocessing

#### Lowercasing

In [8]:
df['clean_text'] = df['customer_message'].str.lower()

print(df[['customer_message', 'clean_text']].head())

                                    customer_message  \
0  I need information about the payment process. ...   
1      I need information about the payment process.   
2  The refund process was fast and convenient. I ...   
3  My refund is still pending and this experience...   
4   Please tell me how to update my account details.   

                                          clean_text  
0  i need information about the payment process. ...  
1      i need information about the payment process.  
2  the refund process was fast and convenient. i ...  
3  my refund is still pending and this experience...  
4   please tell me how to update my account details.  


#### Removing unnecessary symbols or special characters

In [9]:
df['clean_text'] = df['clean_text'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)

pd.set_option('display.max_colwidth', None)
print(df[['customer_message', 'clean_text']].head())


                                                                                               customer_message  \
0  I need information about the payment process. My ticket number is 78732. Please respond as soon as possible.   
1                                                                 I need information about the payment process.   
2                                  The refund process was fast and convenient. I appreciate the quick response.   
3                     My refund is still pending and this experience is frustrating. My ticket number is 33927.   
4                                                              Please tell me how to update my account details.   

                                                                                                  clean_text  
0  i need information about the payment process my ticket number is 78732 please respond as soon as possible  
1                                                               i need information abou

#### Tokenization

In [10]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [11]:

from nltk.tokenize import word_tokenize

df['tokens'] = df['clean_text'].apply(word_tokenize)

print(df[['clean_text', 'tokens']].head())

                                                                                                  clean_text  \
0  i need information about the payment process my ticket number is 78732 please respond as soon as possible   
1                                                               i need information about the payment process   
2                                 the refund process was fast and convenient i appreciate the quick response   
3                    my refund is still pending and this experience is frustrating my ticket number is 33927   
4                                                            please tell me how to update my account details   

                                                                                                                         tokens  
0  [i, need, information, about, the, payment, process, my, ticket, number, is, 78732, please, respond, as, soon, as, possible]  
1                                                                  

#### Remove Stopwords

In [12]:
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

df['filtered_tokens'] = df['tokens'].apply(
    lambda x: [word for word in x if word not in stop_words]
)

print(df[['tokens', 'filtered_tokens']].head())

                                                                                                                         tokens  \
0  [i, need, information, about, the, payment, process, my, ticket, number, is, 78732, please, respond, as, soon, as, possible]   
1                                                                          [i, need, information, about, the, payment, process]   
2                                       [the, refund, process, was, fast, and, convenient, i, appreciate, the, quick, response]   
3                       [my, refund, is, still, pending, and, this, experience, is, frustrating, my, ticket, number, is, 33927]   
4                                                                     [please, tell, me, how, to, update, my, account, details]   

                                                                                 filtered_tokens  
0  [need, information, payment, process, ticket, number, 78732, please, respond, soon, possible]  
1              

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


#### Padding or truncating sequences, if using sequence models


In [13]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer()

tokenizer.fit_on_texts(df['clean_text'])

sequences = tokenizer.texts_to_sequences(df['clean_text'])

padded_sequences = pad_sequences(sequences, maxlen=20)

print(padded_sequences[:5])

[[  0   0   4  29 136  39   1  90  33   3   6   7   2 184  10  12   8  13
    8  14]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   4  29 136  39   1
   90  33]
 [  0   0   0   0   0   0   0   0   1  28  33  11  57   5  58   4  30   1
   31  22]
 [  0   0   0   0   0   3  28   2 120 121   5  20  52   2 122   3   6   7
    2 185]
 [  0   0   0   0   0   0   0   0   0   0   0  10  63  64  65   9  48   3
   45  66]]


## Task 3: Text Vectorization

#### TF-IDF Vectorization


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

x = tfidf.fit_transform(df['clean_text'])

print(x.shape)

(1500, 667)


## Task 4: Baseline Model


#### Logistic Regression with TF-IDF

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

x_train, x_test, y_train, y_test = train_test_split(
    x,
    df['sentiment_label'],
    test_size=0.2,
    random_state=42
)

model = LogisticRegression()

model.fit(x_train, y_train)

y_pred = model.predict(x_test)



#### Create results/model_evaluation.png

In [16]:
import os
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score
import seaborn as sns

os.makedirs('results', exist_ok=True)

accuracy = accuracy_score(y_test, y_pred)

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=['negative', 'neutral', 'positive']
)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['negative', 'neutral', 'positive'],
    yticklabels=['negative', 'neutral', 'positive']
)

plt.title(f'Confusion Matrix\nAccuracy: {accuracy:.2f}')

plt.xlabel('Predicted Label')

plt.ylabel('True Label')

plt.savefig('results/model_evaluation.png', bbox_inches='tight')

plt.close()

print("Improved model_evaluation.png created successfully")
print(classification_report(y_test, y_pred))


Improved model_evaluation.png created successfully
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00       109
     neutral       1.00      1.00      1.00       104
    positive       1.00      1.00      1.00        87

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300



#### Create results/sample_predictions.txt

In [17]:
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

x = tfidf.fit_transform(df['clean_text'])

x_train, x_test, y_train, y_test = train_test_split(
    x,
    df[['customer_message', 'sentiment_label']],
    test_size=0.2,
    random_state=42
)

model = LogisticRegression()

model.fit(x_train, y_train['sentiment_label'])

y_pred = model.predict(x_test)

sample_messages = y_test['customer_message'].iloc[:5].values

actual_labels = y_test['sentiment_label'].iloc[:5].values

predicted_labels = y_pred[:5]

os.makedirs('results', exist_ok=True)

with open('results/sample_predictions.txt', 'w') as file:

    for i in range(5):

        file.write(f"Customer Message:\n{sample_messages[i]}\n")

        file.write(f"Actual Sentiment: {actual_labels[i]}\n")

        file.write(f"Predicted Sentiment: {predicted_labels[i]}\n\n")

print("sample_predictions.txt created successfully")

sample_predictions.txt created successfully


## Task 5: Sequence Model or Conceptual Architecture


#### LSTM Model

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(df['sentiment_label'])

X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences,
    y,
    test_size=0.2,
    random_state=42
)

model = Sequential()

model.add(Embedding(input_dim=5000, output_dim=64, input_length=20))
model.add(LSTM(64))
model.add(Dense(3, activation='softmax'))

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Task 5: Sequence Model – LSTM Architecture
The Sequential LSTM architecture consisted of:
- An Embedding layer for converting tokenized input sequences into dense word vectors.
- An LSTM layer for learning sequential dependencies and contextual information from text data.
- A Dense output layer with softmax activation for multi-class sentiment classification.

The model summary displayed the layer structure, output shapes, and trainable parameters of the sequence model architecture.


#### Input Sequence
Customer messages are converted into numerical token sequences using a tokenizer. These sequences are used as input data for the LSTM model.

#### Embedding Layer
The Embedding layer transformed integer token sequences into dense vector representations, helping the model learn semantic relationships between words.

#### Recurrent/Sequence Layer
The LSTM layer processed the input sequence step-by-step and captured sequential patterns and contextual dependencies present in customer messages.

#### Output Layer
The Dense output layer with softmax activation predicted the probability of the three sentiment classes:
positive, neutral, and negative.

#### Loss Function
The model used sparse_categorical_crossentropy as the loss function because the sentiment labels are multi-class categorical values encoded as integers.

#### Evaluation Metric
Accuracy is used as the evaluation metric to measure the classification performance of the LSTM model.

## Task 6: Attention and Transformer Reflection

#### Why RNNs Struggle with Long-Term Dependencies
RNNs process text sequentially and often struggle to retain information from earlier time steps when handling long sequences. This creates difficulty in learning long-term dependencies within text data.

#### How LSTMs Help with Memory
LSTMs use memory cells and gating mechanisms to preserve important information for longer durations. This helps to reduce the vanishing gradient problem commonly found in traditional RNNs.

#### What Attention Solves in Sequence-to-Sequence Tasks
Attention mechanisms enable the model to focus on the most relevant parts of the input sequence while generating outputs. This improves performance in tasks such as machine translation and text summarization.

#### Why Transformers Are Important in Modern NLP and Generative AI
Transformers process entire sequences in parallel using self-attention mechanisms to capture contextual relationships efficiently. They form the foundation of modern NLP and Generative AI models such as GPT and BERT.